In [5]:
import sys
print(sys.executable)
!{sys.executable} -m pip list

#debugging karena library yg diperlukan blm ada di venv ini. gara2 import dari colab ke jupyterlab (lokal)
!{sys.executable} -m pip install --upgrade --force-reinstall numpy
#solusi lain, copas aja dah ke notebook dari venv lokal

/home/cation-pi/Jupyter/.venv/bin/python3
Package                   Version
------------------------- -----------
anyio                     4.12.1
argon2-cffi               25.1.0
argon2-cffi-bindings      25.1.0
arrow                     1.4.0
asttokens                 3.0.1
async-lru                 2.2.0
attrs                     25.4.0
babel                     2.18.0
beautifulsoup4            4.14.3
bleach                    6.3.0
certifi                   2026.2.25
cffi                      2.0.0
charset-normalizer        3.4.5
comm                      0.2.3
debugpy                   1.8.20
decorator                 5.2.1
defusedxml                0.7.1
executing                 2.2.1
fastjsonschema            2.21.2
fqdn                      1.5.1
h11                       0.16.0
httpcore                  1.0.9
httpx                     0.28.1
idna                      3.11
ipykernel                 7.2.0
ipython                   9.11.0
ipython_pygments_lexers   1.1.1
isodurat

In [7]:
!{sys.executable} -m pip uninstall numpy -y
!{sys.executable} -m pip show numpy

Found existing installation: numpy 2.4.4
Uninstalling numpy-2.4.4:
  Successfully uninstalled numpy-2.4.4


In [6]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.svm import SVC

# =====================
# 1️⃣ Generate Circle Data
# =====================
X, y = make_circles(n_samples=200, factor=0.4, noise=0.05)

# =====================
# ➕ Add HEALTHY center points
# =====================
n_center = 100
center_points = np.random.normal(loc=0.0, scale=0.2, size=(n_center, 2))

# label them as Healthy (1)
center_labels = np.ones(n_center)

# =====================
# Combine datasets
# =====================
X = np.vstack((X, center_points))
y = np.hstack((y, center_labels))

# =====================
# 2️⃣ Map to REAL units
# =====================
bp = 120 + 30 * X[:, 0]
chol = 200 + 50 * X[:, 1]

# =====================
# 3️⃣ Normalize
# =====================
x1 = (bp - 120) / 30
x2 = (chol - 200) / 50
X_norm = np.column_stack((x1, x2))

# =====================
# 4️⃣ Transform
# φ(x) = [r^2, x1]
# =====================
X_transformed = np.zeros_like(X_norm)
X_transformed[:, 0] = x1**2 + x2**2
X_transformed[:, 1] = x1

# =====================
# 5️⃣ Train Models
# =====================
# Nonlinear SVM (original space)
model_nonlinear = SVC(kernel='rbf')
model_nonlinear.fit(X_norm, y)

# Linear SVM (transformed space)
model_linear = SVC(kernel='linear')
model_linear.fit(X_transformed, y)

# =====================
# 6️⃣ Create Subplots
# =====================
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# =====================
# 🔹 1. BEFORE — No Boundary
# =====================
ax = axes[0, 0]
for label_val in [1, 0]:
    ax.scatter(
        bp[y == label_val],
        chol[y == label_val],
        label='Healthy' if label_val == 1 else 'Risky'
    )

ax.set_title("Before Transform (No Boundary)")
ax.set_xlabel("Blood Pressure")
ax.set_ylabel("Cholesterol")
ax.legend()

# =====================
# 🔹 2. BEFORE — With Boundary (RBF)
# =====================
ax = axes[0, 1]

xx, yy = np.meshgrid(
    np.linspace(bp.min()-10, bp.max()+10, 200),
    np.linspace(chol.min()-10, chol.max()+10, 200)
)

gx1 = (xx - 120) / 30
gx2 = (yy - 200) / 50
grid = np.column_stack((gx1.ravel(), gx2.ravel()))

Z = model_nonlinear.predict(grid).reshape(xx.shape)

ax.contourf(xx, yy, Z, alpha=0.3)

for label_val in [1, 0]:
    ax.scatter(
        bp[y == label_val],
        chol[y == label_val],
        label='Healthy' if label_val == 1 else 'Risky'
    )

ax.set_title("Before Transform (Nonlinear Boundary)")
ax.set_xlabel("Blood Pressure")
ax.set_ylabel("Cholesterol")
ax.legend()

# =====================
# 🔹 3. AFTER — No Boundary
# =====================
ax = axes[1, 0]

for label_val in [1, 0]:
    ax.scatter(
        X_transformed[y == label_val, 0],
        X_transformed[y == label_val, 1],
        label='Healthy' if label_val == 1 else 'Risky'
    )

ax.set_title("After Transform (No Boundary)")
ax.set_xlabel("Health Risk Score (r^2)")
ax.set_ylabel("Normalized BP")
ax.legend()

# =====================
# 🔹 4. AFTER — With Boundary (Linear)
# =====================
ax = axes[1, 1]

# Grid in transformed space
z1, z2 = np.meshgrid(
    np.linspace(X_transformed[:,0].min(), X_transformed[:,0].max(), 200),
    np.linspace(X_transformed[:,1].min(), X_transformed[:,1].max(), 200)
)

grid_trans = np.column_stack((z1.ravel(), z2.ravel()))
Z_lin = model_linear.predict(grid_trans).reshape(z1.shape)

ax.contourf(z1, z2, Z_lin, alpha=0.3)

for label_val in [1, 0]:
    ax.scatter(
        X_transformed[y == label_val, 0],
        X_transformed[y == label_val, 1],
        label='Healthy' if label_val == 1 else 'Risky'
    )

ax.set_title("After Transform (Linear Boundary)")
ax.set_xlabel("Health Risk Score (r^2)")
ax.set_ylabel("Normalized BP")
ax.legend()

plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'matplotlib'